In [1]:
# =========================
# FINAL HYBRID + SOC SYSTEM
# =========================

import pandas as pd
import numpy as np
import time
import re
import random
from collections import Counter

# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

train_df["target"] = train_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)
test_df["target"]  = test_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)

# =========================
# PREPROCESSING
# =========================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

drop_cols = ["id","attack_cat","label","4_class","target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["target"]

cat_cols = ["proto","service","state"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train).toarray()
X_test  = scaler.transform(X_test).toarray()

# =========================
# SEQUENCE CREATION
# =========================
def create_sequences(df, X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(1 if np.any(y.iloc[i:i+seq_len].values == 1) else 0)
    return np.array(X_seq), np.array(y_seq)

seq_len = 10
X_train_seq, y_train_seq = create_sequences(train_df, X_train, y_train, seq_len)
X_test_seq, y_test_seq   = create_sequences(test_df, X_test, y_test, seq_len)

# =========================
# TRANSFORMER MODEL
# =========================
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_block(inputs):
    x = MultiHeadAttention(key_dim=64, num_heads=4)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    ffn = Dense(128, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(seq_len, X_train_seq.shape[2]))
x = transformer_block(inputs)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# =========================
# TRAIN
# =========================
start_train = time.time()
model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=64)
training_time = time.time() - start_train
print("Training Time:", training_time)

# =========================
# LOAD RULES
# =========================
def load_rules(filepath):
    rules = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.endswith(","):
                line = line[:-1]
            name, pattern, category = eval(line)
            rules.append((name, re.compile(pattern, re.IGNORECASE), category))
    return rules

rules = load_rules("rules.txt")

# =========================
# TEXT CONVERSION
# =========================
texts = [
    f"{row.proto} {row.service} {row.state} {row.attack_cat}"
    for _, row in test_df.iterrows()
]

# =========================
# HYBRID PREDICTION
# =========================
def hybrid_predict(texts, X_seq):
    results = []
    preds = model.predict(X_seq, verbose=0)

    for i in range(len(X_seq)):
        text = texts[i + seq_len - 1]

        matched = False
        for name, pattern, category in rules:
            if pattern.search(text):
                results.append((1, name, category))
                matched = True
                break

        if not matched:
            results.append((int(preds[i][0] > 0.5), None, None))

    return results

# =========================
# DETECTION TIME
# =========================
start = time.time()
results = hybrid_predict(texts, X_test_seq)
detect_time = time.time() - start

print("Total Detection Time:", detect_time)
print("Per Sequence Time:", detect_time / len(X_test_seq))

# =========================
# SOC LOG GENERATION
# =========================
def random_ip():
    return ".".join(str(random.randint(1,255)) for _ in range(4))

def generate_log(row, pred, rule):
    src = random_ip()
    dst = random_ip()
    proto = row["proto"]
    service = row["service"]

    if pred == 0:
        return f"Normal traffic from {src} to {dst} using {proto}/{service}"
    else:
        return f"⚠️ Attack detected from {src} to {dst} via {proto}/{service} | Rule: {rule}"

# =========================
# BUILD LOGS
# =========================
logs = []
for i, (pred, rule, cat) in enumerate(results[:100]):  # first 100 logs
    row = test_df.iloc[i + seq_len - 1]
    logs.append({
        "text": generate_log(row, pred, rule),
        "attack": "normal" if pred == 0 else "attack",
        "src": random_ip(),
        "dst": random_ip(),
        "proto": row["proto"],
        "service": row["service"]
    })

# =========================
# EXPLAINABILITY
# =========================
def explain(batch):
    attacks = [x for x in batch if x["attack"] != "normal"]

    if not attacks:
        print("No attacks")
        return

    types = Counter([x["attack"] for x in attacks])
    print("Attack Summary:", types)

    top = types.most_common(1)[0]
    print("Top Attack:", top)

# =========================
# RUN SOC ANALYSIS
# =========================
print("\n--- SAMPLE LOGS ---")
for l in logs[:10]:
    print(l["text"])

print("\n--- SOC ANALYSIS ---")
explain(logs)

KeyboardInterrupt: 

In [ ]:
import sys
print(sys.executable)

c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\python.exe


In [ ]:
import sys
!{sys.executable} -m pip install transformers torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------- ----------- 7.1/10.1 MB 36.3 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 35.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   -- ------------------------------------- 6.8/114.5 MB 35.0 MB/s eta 0:00:04
   ----- ---------------------------------- 16.0/114.5 MB 38.7 MB/s eta 0:00:03
   -------- ------------------------------- 25.2/114.5 MB 40.9 MB/s eta 0:00:03
   ----------- ---------------------------- 34.1/114.5 MB 41.6 MB/s eta 0:00:02
   ------------- -------------------------- 39.3/114.5 MB 37.9 MB/s eta 0:00:02
   ---------------- ----------------------- 48.5/114.5 MB 38.6 MB/s eta 0:00:02
   -------------------- -----------


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install transformers torch

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [ ]:
pip uninstall transformers -y

Found existing installation: transformers 5.4.0
Uninstalling transformers-5.4.0:
  Successfully uninstalled transformers-5.4.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install transformers==4.36.2

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------- ----------------- 4.7/8.2 MB 23.7 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 21.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/566.4 kB ? eta -:--:--
   --------------------------------------- 566.4/566.4 kB 20.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 30.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =========================
# FINAL HYBRID + SOC SYSTEM
# =========================

import pandas as pd
import numpy as np
import time
import re
import random
from collections import Counter

# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

train_df["target"] = train_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)
test_df["target"]  = test_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)

# =========================
# PREPROCESSING
# =========================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

drop_cols = ["id","attack_cat","label","4_class","target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["target"]

cat_cols = ["proto","service","state"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train).toarray()
X_test  = scaler.transform(X_test).toarray()

# =========================
# SEQUENCES
# =========================
def create_sequences(df, X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(1 if np.any(y.iloc[i:i+seq_len].values == 1) else 0)
    return np.array(X_seq), np.array(y_seq)

seq_len = 10
X_train_seq, y_train_seq = create_sequences(train_df, X_train, y_train, seq_len)
X_test_seq, y_test_seq   = create_sequences(test_df, X_test, y_test, seq_len)

# =========================
# MODEL
# =========================
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_block(inputs):
    x = MultiHeadAttention(key_dim=64, num_heads=4)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    ffn = Dense(128, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(seq_len, X_train_seq.shape[2]))
x = transformer_block(inputs)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# =========================
# TRAIN
# =========================
start_train = time.time()

model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=64)

training_time = time.time() - start_train
print(f"\nTraining Time: {training_time:.2f} seconds")

# =========================
# RULES
# =========================
def load_rules(filepath):
    rules = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.endswith(","):
                line = line[:-1]
            name, pattern, category = eval(line)
            rules.append((name, re.compile(pattern, re.IGNORECASE), category))
    return rules

rules = load_rules("rules.txt")

# =========================
# TEXTS
# =========================
texts = [
    f"{row.proto} {row.service} {row.state} {row.attack_cat}"
    for _, row in test_df.iterrows()
]

# =========================
# HYBRID PREDICTION
# =========================
preds = model.predict(X_test_seq, verbose=0)

hybrid_results = []
for i in range(len(X_test_seq)):
    text = texts[i + seq_len - 1]

    matched = False
    for name, pattern, cat in rules:
        if pattern.search(text):
            hybrid_results.append((1, "rule", name, cat))
            matched = True
            break

    if not matched:
        label = int(preds[i][0] > 0.5)
        hybrid_results.append((label, "dl", None, None))

# =========================
# DETECTION TIME
# =========================
start = time.time()
_ = model.predict(X_test_seq[:1000], verbose=0)
end = time.time()

print("Detection Time per Sequence:", (end-start)/1000)

# =========================
# CONFUSION MATRIX
# =========================
from sklearn.metrics import confusion_matrix
y_pred = [x[0] for x in hybrid_results]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_seq, y_pred))

# =========================
# RANDOM SEQUENCE + SOC
# =========================
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def random_ip():
    return ".".join(str(random.randint(1,255)) for _ in range(4))

def generate_log(row, pred):
    src = random_ip()
    dst = random_ip()
    proto = row["proto"]
    service = row["service"]

    if pred == 0:
        return {
            "text": f"Normal traffic from {src} to {dst} via {proto}/{service}",
            "attack": "normal",
            "src": src,
            "dst": dst,
            "proto": proto,
            "service": service
        }
    else:
        attack = row["attack_cat"].lower()
        return {
            "text": f"{attack} attack from {src} to {dst} via {proto}/{service}",
            "attack": attack,
            "src": src,
            "dst": dst,
            "proto": proto,
            "service": service
        }

# pick random sequence
i = random.randint(0, len(X_test_seq)-1)

print("\n🔍 RANDOM SEQUENCE:", i)

sequence_logs = []
for j in range(i, i+seq_len):
    row = test_df.iloc[j]
    pred = hybrid_results[j][0]
    sequence_logs.append(generate_log(row, pred))

# =========================
# SUMMARY
# =========================
combined = " ".join([l["text"] for l in sequence_logs])[:1024]

summary = summarizer(combined, max_length=100, min_length=50, do_sample=False)

print("\n🧠 SUMMARY:")
print(summary[0]["summary_text"])

# =========================
# EXPLAINABILITY
# =========================
def explain(batch):
    attacks = [x for x in batch if x["attack"] != "normal"]

    if not attacks:
        print("No attacks detected")
        return

    attack_types = Counter([x["attack"] for x in attacks])
    top_attack = attack_types.most_common(1)[0][0]

    src = Counter([x["src"] for x in attacks]).most_common(1)[0][0]
    dst = Counter([x["dst"] for x in attacks]).most_common(1)[0][0]

    print("\n🔍 EXPLAINABILITY:")
    print("WHO:", src)
    print("WHERE:", dst)
    print("WHAT:", top_attack)

    if len(attacks) > 5:
        print("🚨 THREAT LEVEL: HIGH")
    else:
        print("🚨 THREAT LEVEL: LOW")

    print("\n🛡️ ACTION:")
    print(f"Block IP {src}, monitor {top_attack} activity")

explain(sequence_logs)

Epoch 1/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 115s 40ms/step - accuracy: 0.9916 - loss: 0.0241
Epoch 2/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 107s 39ms/step - accuracy: 0.9936 - loss: 0.0185
Epoch 3/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 105s 38ms/step - accuracy: 0.9945 - loss: 0.0160

Training Time: 328.63 seconds
Detection Time per Sequence: 0.000367201566696167

Confusion Matrix:
[[35602  1370]
 [  162 45188]]


c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecat


🔍 RANDOM SEQUENCE: 55234

🧠 SUMMARY:
Attack from 30.44.205.98 to 245.182.51.160 via udp/dns dos attack from 122.52.24.186 to 162.238.245.237 via ipv6/- generic attack. generic attack from 44.112.237.105 to 114.57.164.192 via uDP/Dns generic attack and exploit. attack.

🔍 EXPLAINABILITY:
WHO: 44.112.237.105
WHERE: 114.57.164.192
WHAT: generic
🚨 THREAT LEVEL: HIGH

🛡️ ACTION:
Block IP 44.112.237.105, monitor generic activity


In [1]:
# =========================
# FINAL HYBRID + SOC SYSTEM (SELF-LEARNING)
# =========================

import pandas as pd
import numpy as np
import time
import re
import random
from collections import Counter

# =========================
# LOAD DATA
# =========================
train_df = pd.read_csv("UNSW_NB15_grouped_training_class.csv")
test_df  = pd.read_csv("UNSW_NB15_grouped_testing_class.csv")

train_df["target"] = train_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)
test_df["target"]  = test_df["4_class"].apply(lambda x: 0 if x == "Benign" else 1)

# =========================
# PREPROCESSING
# =========================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

drop_cols = ["id","attack_cat","label","4_class","target"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["target"]

X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df["target"]

cat_cols = ["proto","service","state"]

preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

scaler = StandardScaler(with_mean=False)
X_train = scaler.fit_transform(X_train).toarray()
X_test  = scaler.transform(X_test).toarray()

# =========================
# SEQUENCES
# =========================
def create_sequences(df, X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(1 if np.any(y.iloc[i:i+seq_len].values == 1) else 0)
    return np.array(X_seq), np.array(y_seq)

seq_len = 10
X_train_seq, y_train_seq = create_sequences(train_df, X_train, y_train, seq_len)
X_test_seq, y_test_seq   = create_sequences(test_df, X_test, y_test, seq_len)

# =========================
# MODEL (TRANSFORMER)
# =========================
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_block(inputs):
    x = MultiHeadAttention(key_dim=64, num_heads=4)(inputs, inputs)
    x = LayerNormalization()(x + inputs)
    ffn = Dense(128, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization()(x + ffn)

inputs = Input(shape=(seq_len, X_train_seq.shape[2]))
x = transformer_block(inputs)
x = transformer_block(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# =========================
# TRAIN
# =========================
start_train = time.time()
model.fit(X_train_seq, y_train_seq, epochs=3, batch_size=64)
print("Training Time:", time.time() - start_train)

# =========================
# RULE HANDLING
# =========================
def load_rules(filepath):
    rules = []
    try:
        with open(filepath, "r") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                if line.endswith(","):
                    line = line[:-1]
                name, pattern, category = eval(line)
                rules.append((name, re.compile(pattern, re.IGNORECASE), category))
    except FileNotFoundError:
        open(filepath, "w").close()
    return rules

def append_rule(filepath, name, pattern, category):
    with open(filepath, "a") as f:
        f.write(f'("{name}", r"{pattern}", "{category}"),\n')

def rule_exists(rules, pattern):
    for _, p, _ in rules:
        if p.pattern == pattern:
            return True
    return False

rules = load_rules("rules.txt")

# =========================
# TEXT FORMAT
# =========================
texts = [
    f"{row.proto} {row.service} {row.state} {row.attack_cat}"
    for _, row in test_df.iterrows()
]

# =========================
# HYBRID SYSTEM + RULE LEARNING
# =========================
preds = model.predict(X_test_seq, verbose=0)

hybrid_results = []

for i in range(len(X_test_seq)):
    row = test_df.iloc[i + seq_len - 1]
    text = texts[i + seq_len - 1]

    matched = False

    # RULE CHECK
    for name, pattern, cat in rules:
        if pattern.search(text):
            hybrid_results.append((1, "rule", name, cat))
            matched = True
            break

    if not matched:
        label = int(preds[i][0] > 0.5)

        if label == 1:
            attack = row["attack_cat"].lower()

            # 🔥 richer pattern (proto + service + state + attack)
            pattern = f"\\b{row['proto']}.*{row['service']}.*{row['state']}.*{attack}\\b"
            rule_name = f"dl_{attack}_{i}"

            if not rule_exists(rules, pattern):
                append_rule("rules.txt", rule_name, pattern, attack)
                rules.append((rule_name, re.compile(pattern, re.IGNORECASE), attack))

            hybrid_results.append((1, "dl_rule_added", rule_name, attack))

        else:
            hybrid_results.append((0, "dl", None, None))

# =========================
# DETECTION TIME
# =========================
start = time.time()
_ = model.predict(X_test_seq[:1000], verbose=0)
print("Detection Time per Sequence:", (time.time() - start)/1000)

# =========================
# CONFUSION MATRIX
# =========================
from sklearn.metrics import confusion_matrix

y_pred = [x[0] for x in hybrid_results]

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_seq, y_pred))

# =========================
# SOC LOG GENERATION
# =========================
from transformers import pipeline
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def random_ip():
    return ".".join(str(random.randint(1,255)) for _ in range(4))

def generate_log(row, pred):
    src = random_ip()
    dst = random_ip()

    if pred == 0:
        return {"text": f"Normal traffic {src}->{dst} {row.proto}/{row.service}",
                "attack": "normal", "src": src, "dst": dst}
    else:
        attack = row["attack_cat"].lower()
        return {"text": f"{attack} attack {src}->{dst} {row.proto}/{row.service}",
                "attack": attack, "src": src, "dst": dst}

# =========================
# RANDOM SEQUENCE
# =========================
i = random.randint(0, len(X_test_seq)-1)
sequence_logs = []

for j in range(i, i+seq_len):
    row = test_df.iloc[j]
    pred = hybrid_results[j][0]
    sequence_logs.append(generate_log(row, pred))

# =========================
# SUMMARY
# =========================
combined = " ".join([l["text"] for l in sequence_logs])[:1024]
summary = summarizer(combined, max_length=100, min_length=50, do_sample=False)

print("\nSUMMARY:")
print(summary[0]["summary_text"])

# =========================
# EXPLAINABILITY
# =========================
def explain(batch):
    attacks = [x for x in batch if x["attack"] != "normal"]

    if not attacks:
        print("No attacks detected")
        return

    attack_types = Counter([x["attack"] for x in attacks])
    top_attack = attack_types.most_common(1)[0][0]

    src = Counter([x["src"] for x in attacks]).most_common(1)[0][0]
    dst = Counter([x["dst"] for x in attacks]).most_common(1)[0][0]

    print("\nEXPLAINABILITY:")
    print("WHO:", src)
    print("WHERE:", dst)
    print("WHAT:", top_attack)

    print("THREAT LEVEL:", "HIGH" if len(attacks) > 5 else "LOW")
    print("ACTION: Block IP", src)

explain(sequence_logs)

Epoch 1/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 270s 94ms/step - accuracy: 0.9907 - loss: 0.0257
Epoch 2/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 292s 106ms/step - accuracy: 0.9940 - loss: 0.0181
Epoch 3/3
2740/2740 ━━━━━━━━━━━━━━━━━━━━ 296s 97ms/step - accuracy: 0.9937 - loss: 0.0178
Training Time: 862.6248779296875
Detection Time per Sequence: 0.0014521775245666503

Confusion Matrix:
[[ 7328 29644]
 [    3 45347]]


c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Kruthika\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download


SUMMARY:
 exploits attack 131.185.132.43->80.149.182.184 snp/- exploits attack 44.251.118.42->20.168.18.93 a/n/- shellcode attack 29.252.114.163->6.196.166.253 tcp/ftp-data dos attack 160.90.84.9->139.137.139.7 ipcomp/- dos attack 48.136.144.43 ->235.

EXPLAINABILITY:
WHO: 131.185.132.43
WHERE: 80.149.182.184
WHAT: exploits
THREAT LEVEL: HIGH
ACTION: Block IP 131.185.132.43
